# 🧮 Calc Groups Cathedral — Seed Notebook

> 🏗️ Build: **2026-05-29 12:41:15** &nbsp;·&nbsp; if you don't see this stamp after re-upload, close the notebook tab and reopen it.

> **Run this notebook ONCE to bootstrap the game.**
> It populates the `Cathedral_LH` Lakehouse with synthetic Sales / Date / Customer / Budget tables (3 years),
> then creates the `Cathedral_Model` Direct Lake semantic model on top of them.

## Requirements
1. Attach **`Cathedral_LH`** as the default Lakehouse on this notebook (📚 icon in the left rail → *Add* → Existing Lakehouse).
2. Run all cells top → bottom.

Total runtime: ~1–2 minutes.


In [ ]:
# === Imports & constants ===
# Upgrade PyJWT silently to avoid sempy-labs dependency conflict warning later on.
# NOTE: stderr is captured so the cosmetic 'pip's dependency resolver' warning is hidden.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "--disable-pip-version-check", "PyJWT>=2.6.0"],
               check=False, capture_output=True)

import random, datetime as dt
from pyspark.sql import functions as F
from pyspark.sql import types as T

SEED_VERSION = "v1"          # bump if you change generators
RNG_SEED = 42                # deterministic
YEARS = [2023, 2024, 2025]
N_CUSTOMERS = 50
N_SALES = 100_000

random.seed(RNG_SEED)

# Best-effort: detect attached Lakehouse (don't hard-fail, just warn).
lh_name = ""
for key in ["trident.lakehouse.name", "trident.activeworkspace.lakehouseName",
            "trident.workspace.lakehouseName"]:
    try:
        v = spark.conf.get(key)
        if v: lh_name = v; break
    except Exception:
        pass
print(f"Default Lakehouse: {lh_name or '(not detected)'}")

# Drop stale tables from prior runs (idempotent).
for t in ["date", "customer", "sales", "budget",
          "Date", "Customer", "Sales", "Budget"]:
    try:
        spark.sql(f"DROP TABLE IF EXISTS {t}")
    except Exception:
        pass
print("🧹 Clean slate.")


## Step 1 — Date dimension (3 years, daily grain)


In [ ]:
start = dt.date(YEARS[0], 1, 1)
end   = dt.date(YEARS[-1], 12, 31)
days  = (end - start).days + 1

rows = []
for i in range(days):
    d = start + dt.timedelta(days=i)
    rows.append((
        int(d.strftime("%Y%m%d")),               # DateKey
        d,                                       # Date
        d.year,                                  # Year
        (d.month - 1) // 3 + 1,                  # Quarter
        d.month,                                 # MonthNum
        d.strftime("%B"),                        # MonthName
        d.day,                                   # DayOfMonth
        d.strftime("%A"),                        # DayName
        d.isoweekday() in (6, 7),                # IsWeekend
    ))

schema = T.StructType([
    T.StructField("DateKey",     T.IntegerType(),   False),
    T.StructField("Date",        T.DateType(),      False),
    T.StructField("Year",        T.IntegerType(),   False),
    T.StructField("Quarter",     T.IntegerType(),   False),
    T.StructField("MonthNum",    T.IntegerType(),   False),
    T.StructField("MonthName",   T.StringType(),    False),
    T.StructField("DayOfMonth",  T.IntegerType(),   False),
    T.StructField("DayName",     T.StringType(),    False),
    T.StructField("IsWeekend",   T.BooleanType(),   False),
])

date_df = spark.createDataFrame(rows, schema)
(date_df.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("date"))
print(f"✅ date  rows={date_df.count():,}")


## Step 2 — Customer dimension


In [ ]:
REGIONS  = ["EU-North", "EU-South", "US-East", "US-West", "APAC"]
SEGMENTS = ["Enterprise", "SMB", "Consumer"]

rng = random.Random(RNG_SEED + 1)
rows = []
for i in range(1, N_CUSTOMERS + 1):
    rows.append((
        i,
        f"Customer {i:03d}",
        rng.choice(REGIONS),
        rng.choice(SEGMENTS),
    ))

cust_df = spark.createDataFrame(rows, ["CustomerKey", "CustomerName", "Region", "Segment"])
(cust_df.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("customer"))
print(f"✅ customer  rows={cust_df.count():,}")


## Step 3 — Sales fact (~100k rows, deterministic, with seasonality)


In [ ]:
# Build a date list once (Python side) for fast sampling
date_pool = [(int(d.strftime('%Y%m%d')), d) for d in
             (dt.date(YEARS[0], 1, 1) + dt.timedelta(days=i) for i in range(days))]

rng = random.Random(RNG_SEED + 2)

# Mild seasonality: Q4 +30%, Q1 -10%
def season_factor(month):
    if month in (10, 11, 12): return 1.30
    if month in (1, 2):       return 0.90
    return 1.0

# YoY growth: 2023 base, 2024 +12%, 2025 +18%
year_growth = {2023: 1.00, 2024: 1.12, 2025: 1.18}

rows = []
for sales_key in range(1, N_SALES + 1):
    date_key, d = rng.choice(date_pool)
    cust_key = rng.randint(1, N_CUSTOMERS)
    qty = rng.randint(1, 20)
    unit_price = round(rng.uniform(10.0, 500.0), 2)
    raw_amount = qty * unit_price
    amount = round(raw_amount * season_factor(d.month) * year_growth[d.year], 2)
    rows.append((sales_key, date_key, cust_key, qty, unit_price, amount))

schema = T.StructType([
    T.StructField("SalesKey",   T.IntegerType(),   False),
    T.StructField("DateKey",    T.IntegerType(),   False),
    T.StructField("CustomerKey", T.IntegerType(),  False),
    T.StructField("Quantity",   T.IntegerType(),   False),
    T.StructField("UnitPrice",  T.DoubleType(),    False),
    T.StructField("Amount",     T.DoubleType(),    False),
])

sales_df = spark.createDataFrame(rows, schema)
(sales_df.write.format("delta").mode("overwrite")
         .option("overwriteSchema", "true")
         .saveAsTable("sales"))
print(f"✅ sales  rows={sales_df.count():,}")


## Step 4 — Budget (monthly per region, planned ≈ 95% of actuals)


In [ ]:
# Budget needs Year/MonthNum/Region — join back to date+customer
actuals = (sales_df
    .join(date_df, "DateKey")
    .join(cust_df, "CustomerKey")
    .groupBy("Year", "MonthNum", "Region")
    .agg(F.round(F.sum("Amount"), 2).alias("Actual")))

budget_df = actuals.withColumn("Budget", F.round(F.col("Actual") * 0.95, 2)).select(
    "Year", "MonthNum", "Region", "Budget"
)

(budget_df.write.format("delta").mode("overwrite")
          .option("overwriteSchema", "true")
          .saveAsTable("budget"))
print(f"✅ budget  rows={budget_df.count():,}")


## Step 5 — Summary


In [ ]:
for t in ["date", "customer", "sales", "budget"]:
    n = spark.table(t).count()
    print(f"  {t:<10} {n:>10,} rows")
print("\n🏛️  Lakehouse Cathedral_LH is ready.")


## Step 6 — Create the `Cathedral_Model` semantic model (Direct Lake)

Uses `sempy-labs` to build a Direct Lake semantic model on top of the 4 tables.
Idempotent: if the model already exists, it's left alone (delete it first if you want a fresh one).


In [ ]:
# Install / import sempy-labs (Fabric runtime ships sempy but not labs by default)
import subprocess, sys, importlib
try:
    import sempy_labs as labs
    from sempy_labs import directlake as labs_dl  # explicit submodule import
except ImportError:
    print("Installing semantic-link-labs...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "semantic-link-labs"],
                   check=True)
    importlib.invalidate_caches()
    import sempy_labs as labs
    from sempy_labs import directlake as labs_dl
print(f"sempy-labs version: {getattr(labs, '__version__', '?')}")

import sempy.fabric as fabric

MODEL_NAME = "Cathedral_Model"
LAKEHOUSE  = "Cathedral_LH"
# Physical tables in OneLake are lowercase (Spark's saveAsTable behavior).
# Dict maps semantic-model table name (Pascal) → physical Delta folder name (lowercase).
# Logical name (Pascal, used in DAX) -> physical Delta folder (lowercase, Spark default).
TABLES = {
    "Date":     "date",
    "Customer": "customer",
    "Sales":    "sales",
    "Budget":   "budget",
}

# Check existence via sempy.fabric.list_datasets()
try:
    df = fabric.list_datasets()
    # Column name varies across versions: 'Dataset Name', 'name', etc.
    name_col = next((c for c in df.columns if c.lower() in ("dataset name", "name", "display name")), None)
    already = (df[name_col] == MODEL_NAME).any() if name_col else False
except Exception as e:
    print(f"(could not list datasets: {e}); will try to create...")
    already = False

if already:
    print(f"✅ Semantic model '{MODEL_NAME}' already exists — attempting refresh only.")
    import time
    last_err = None
    for attempt in range(1, 6):
        try:
            fabric.refresh_dataset(dataset=MODEL_NAME, refresh_type="full")
            print(f"✅ Refresh succeeded on attempt {attempt}.")
            last_err = None
            break
        except Exception as e:
            last_err = e
            print(f"   attempt {attempt} failed: {e}")
            time.sleep(20)
    if last_err is not None:
        print("⚠️  Refresh still failing. Delete the model in Fabric UI and re-run this cell to recreate from scratch.")
        print(f"   Last error: {last_err}")
else:
    # Warm up SQL endpoint metadata for the freshly-written tables.
    # Without this wait, refresh fails with "tables don't exist or access denied".
    import time
    print("⏳ Waiting 45s for SQL endpoint metadata sync of newly-written tables...")
    time.sleep(45)

    print(f"⏳ Creating Direct Lake semantic model '{MODEL_NAME}' from Lakehouse '{LAKEHOUSE}' (without refresh)...")
    labs_dl.generate_direct_lake_semantic_model(
        dataset=MODEL_NAME,
        tables=TABLES,
        source=LAKEHOUSE,
        source_type="Lakehouse",
        refresh=False,
    )
    print(f"✅ Created '{MODEL_NAME}' (unrefreshed).")

    # Refresh separately with retries; first refresh on a brand new Direct Lake model
    # can flake while the SQL endpoint catches up.
    print("⏳ Refreshing model (with retries)...")
    last_err = None
    for attempt in range(1, 6):
        try:
            fabric.refresh_dataset(dataset=MODEL_NAME, refresh_type="full")
            print(f"✅ Refresh succeeded on attempt {attempt}.")
            last_err = None
            break
        except Exception as e:
            last_err = e
            print(f"   attempt {attempt} failed: {e}")
            time.sleep(20)
    if last_err is not None:
        print("⚠️  Refresh still failing — model exists, you can retry refresh manually from Fabric UI.")
        print(f"   Last error: {last_err}")


## Step 7 — Add relationships + seed measure

The judge needs at minimum:
- relationship `Sales[DateKey]   ↔ Date[DateKey]`
- relationship `Sales[CustomerKey] ↔ Customer[CustomerKey]`
- one seed measure `Sales Amount Seed := SUM(Sales[Amount])` (excluded from your elegance score)


In [ ]:
# Ensure sempy-labs is available (re-import is cheap if Step 6 already installed it)
import subprocess, sys, importlib
try:
    import sempy_labs as labs
    from sempy_labs import tom as labs_tom  # EXPLICIT submodule import
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "semantic-link-labs"],
                   check=True)
    importlib.invalidate_caches()
    import sempy_labs as labs
    from sempy_labs import tom as labs_tom
import traceback

MODEL_NAME = "Cathedral_Model"
SEED_MEASURE = "Sales Amount Seed"

def _safe(label, fn):
    try:
        fn()
        print(f"  ✅ {label}")
    except Exception as e:
        print(f"  ❌ {label}: {type(e).__name__}: {e}")
        traceback.print_exc()

print(f"🔌 Connecting to semantic model {MODEL_NAME!r}...")
try:
    ctx = labs_tom.connect_semantic_model(dataset=MODEL_NAME, readonly=False)
except Exception:
    print("❌ connect_semantic_model failed")
    traceback.print_exc()
    raise

with ctx as tom:

    # --- Inventory check (each piece isolated) ---
    try:
        tnames = [t.Name for t in tom.model.Tables]
        print("📋 Tables in model:", tnames)
    except Exception:
        print("❌ enumerate Tables failed")
        traceback.print_exc()
        tnames = []

    for tname in ("Date", "Customer", "Sales", "Budget"):
        try:
            cols = [c.Name for c in tom.model.Tables[tname].Columns]
            print(f"   {tname}: {cols}")
        except Exception as e:
            print(f"   ❌ Table {tname!r} not found: {e}")

    # --- relationships ---
    def has_rel(from_tbl, from_col, to_tbl, to_col):
        for r in tom.model.Relationships:
            try:
                if (r.FromTable.Name == from_tbl and r.FromColumn.Name == from_col
                    and r.ToTable.Name == to_tbl and r.ToColumn.Name == to_col):
                    return True
            except Exception:
                pass
        return False

    def _add_rel_date():
        if has_rel("Sales", "DateKey", "Date", "DateKey"):
            print("  = rel Sales->Date already exists"); return
        tom.add_relationship(
            from_table="Sales", from_column="DateKey",
            to_table="Date",  to_column="DateKey",
            from_cardinality="Many", to_cardinality="One",
            cross_filtering_behavior="OneDirection",
        )

    def _add_rel_cust():
        if has_rel("Sales", "CustomerKey", "Customer", "CustomerKey"):
            print("  = rel Sales->Customer already exists"); return
        tom.add_relationship(
            from_table="Sales", from_column="CustomerKey",
            to_table="Customer", to_column="CustomerKey",
            from_cardinality="Many", to_cardinality="One",
            cross_filtering_behavior="OneDirection",
        )

    def _mark_date():
        tom.mark_as_date_table(table_name="Date", column_name="Date")

    def _add_seed_measure():
        if any(m.Name == SEED_MEASURE for m in tom.model.Tables["Sales"].Measures):
            print(f"  = measure {SEED_MEASURE} already exists"); return
        tom.add_measure(
            table_name="Sales",
            measure_name=SEED_MEASURE,
            expression="SUM(Sales[Amount])",
            format_string="#,##0.00",
            description="Seed measure — excluded from elegance score.",
        )

    _safe("relationship Sales[DateKey] -> Date[DateKey]", _add_rel_date)
    _safe("relationship Sales[CustomerKey] -> Customer[CustomerKey]", _add_rel_cust)
    _safe("mark Date as date table (column 'Date')", _mark_date)
    _safe(f"add measure {SEED_MEASURE}", _add_seed_measure)

print("✅ Model committed.")
print()
print("🎉 Setup complete. Open the `CalcGroups_Cathedral` notebook to start playing!")
